# F1 Race Strategy - Feature Engineering
This notebook covers the feature engineering phase for the Formula 1 datasets. We create new layout-based features for circuits, compute historical driver and team performance indicators, and prepare the final datasets for the race strategy optimizer.

In [1]:
import pandas as pd
import numpy as np

circuits_path = 'f1_circuits_cleaned.csv'
results_path = 'f1_race_results_cleaned.csv'

circuits_df = pd.read_csv(circuits_path)
results_df = pd.read_csv(results_path)

## 1. Circuit-Level Feature Engineering
We create physical layout metrics that will help the simulation engine assess track difficulty and pit stop strategies:
* **Turn Density (`turns_per_km`)**: Number of turns divided by sirkuit length.
* **Elevation Change Intensity (`elevation_intensity`)**: Vertial height change in meters per kilometer.
* **Pit Stop Transit Overhead (`pit_stop_overhead_sec`)**: Estimated transit time loss in seconds based on the pit lane length (using an 80 km/h speed limit + 2.5 seconds tyre change time).
* **Track Speed Category (`track_speed_category`)**: Classified as Low, Medium, or High Speed track based on historical average top speed.

In [2]:
# Turn Density (Turns per Kilometer)
circuits_df['turns_per_km'] = circuits_df['number_of_turns'] / circuits_df['track_length_km']

# Elevation Change Intensity (Elevation Change per Kilometer)
circuits_df['elevation_intensity'] = circuits_df['elevation_change_m'] / circuits_df['track_length_km']

# Estimated Pit Stop Overhead Time (in seconds)
# pit_lane_length_m / (80 km/h converted to m/s) + 2.5s tyre change time
circuits_df['pit_stop_overhead_sec'] = (circuits_df['pit_lane_length_m'] / (80 / 3.6)) + 2.5

# Categorize Track Speed based on top speed
def categorize_speed(speed):
    if speed < 300:
        return 'Low Speed'
    elif speed <= 340:
        return 'Medium Speed'
    else:
        return 'High Speed'

circuits_df['track_speed_category'] = circuits_df['avg_top_speed_kmh'].apply(categorize_speed)

circuits_df[['circuit_name', 'turns_per_km', 'elevation_intensity', 'pit_stop_overhead_sec', 'track_speed_category']].head()

,circuit_name,turns_per_km,elevation_intensity,pit_stop_overhead_sec,track_speed_category
0,Mugello Circuit,4.242321,6.278636,14.605,Low Speed
1,Circuit de Monaco,2.507523,26.830491,20.950,Low Speed
2,Circuit de Spa-Francorchamps,1.364050,19.854501,23.425,High Speed
3,Albert Park Circuit - Revised 2020,4.568024,43.495531,18.475,Medium Speed
4,Marina Bay Street Circuit,3.357693,32.984397,16.225,Medium Speed


## 2. Driver & Team Performance Index
We calculate the historical average performance metrics for drivers and teams to act as baseline skill inputs for the simulation engine:
* **`driver_avg_finish_pos`**: A driver's long-term average finish rank.
* **`team_avg_points`**: A team's average points scored per race.

In [3]:
# Driver historical average finish position
driver_stats = results_df.groupby('driver_name')['finish_position'].mean().reset_index()
driver_stats.columns = ['driver_name', 'driver_avg_finish_pos']
results_df = pd.merge(results_df, driver_stats, on='driver_name', how='left')

# Team historical average points scored
team_stats = results_df.groupby('team_name')['points_scored'].mean().reset_index()
team_stats.columns = ['team_name', 'team_avg_points']
results_df = pd.merge(results_df, team_stats, on='team_name', how='left')

# Verify driver and team indexes
results_df[['driver_name', 'team_name', 'driver_avg_finish_pos', 'team_avg_points']].drop_duplicates().reset_index(drop=True)

,driver_name,team_name,driver_avg_finish_pos,team_avg_points
0,Logan Sargeant,Williams,10.693333,4.887500
1,Charles Leclerc,Ferrari,10.732394,4.600000
2,Fernando Alonso,Aston Martin,12.407407,3.517857
3,Max Verstappen,Red Bull Racing,11.303030,3.361111
4,George Russell,Mercedes,11.063492,4.753425
5,Pierre Gasly,Alpine,11.676056,2.973684
6,Sergio Perez,Red Bull Racing,12.029851,3.361111
7,Yuki Tsunoda,AlphaTauri,10.666667,3.791667
8,Carlos Sainz,Ferrari,10.695652,4.600000
9,Nico Hulkenberg,Haas,10.734375,5.073529


## 3. Save Engineered Datasets
We save the newly engineered datasets to the workspace as CSV files.

In [4]:
engineered_circuits_path = 'f1_circuits_engineered.csv'
engineered_results_path = 'f1_race_results_engineered.csv'

circuits_df.to_csv(engineered_circuits_path, index=False, encoding='utf-8')
results_df.to_csv(engineered_results_path, index=False, encoding='utf-8')

print("Engineered datasets successfully saved to the workspace.")

Engineered datasets successfully saved to the workspace.
